# Fine-Tuning Qwen3-4B + LoRA — Prompt Expander (Tim 4)

Standard HuggingFace Transformers + PEFT (tanpa Unsloth).
Kompatibel dengan PyTorch 2.4.1 + PEFT 0.14+ di SR4.


## 1. Install Dependencies

In [ ]:
!pip install transformers==5.5.4 peft==0.17.1 trl==1.1.0 accelerate bitsandbytes datasets -q


## 2. Imports & Konfigurasi

In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ============================================================
# 🔧 SET PATH DI SINI — ubah sesuai lokasi folder kamu
# ============================================================
CUSTOM_BASE_PATH   = "/home/jovyan/SR4"
CUSTOM_DATASET_DIR = "/home/jovyan/SR4/DATASET"
CUSTOM_OUTPUT_DIR  = "/home/jovyan/SR4/output"

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
if IN_COLAB and CUSTOM_BASE_PATH and "/drive/" in CUSTOM_BASE_PATH:
    from google.colab import drive
    drive.mount("/content/drive")

BASE_PATH   = CUSTOM_BASE_PATH or ("/content" if IN_COLAB else os.getcwd())
DATASET_DIR = CUSTOM_DATASET_DIR or os.path.join(BASE_PATH, "dataset")
OUTPUT_DIR  = CUSTOM_OUTPUT_DIR  or os.path.join(BASE_PATH, "output")

assert os.path.exists(BASE_PATH),   f"❌ BASE_PATH tidak ditemukan: {BASE_PATH}"
assert os.path.exists(DATASET_DIR), f"❌ DATASET_DIR tidak ditemukan: {DATASET_DIR}"

print(f"📍 Base Path   : {BASE_PATH}")
print(f"📂 Dataset Dir : {DATASET_DIR}")
print(f"💾 Output Dir  : {OUTPUT_DIR}")
print(f"🌐 Environment : {'Google Colab' if IN_COLAB else 'Local/SR4'}")
print(f"🔥 CUDA        : {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


In [ ]:
class Config:
    # === Model ===
    MODEL_NAME      = "Qwen/Qwen3-4B"
    MAX_SEQ_LENGTH  = 2048
    DTYPE           = torch.bfloat16
    SEED            = 42

    # === LoRA — diturunkan untuk cegah overfitting ===
    LORA_R          = 16      # dari 64 → lebih kecil, generalisasi lebih baik
    LORA_ALPHA      = 32      # 2x rank (standar)
    LORA_DROPOUT    = 0.15    # dinaikkan sedikit untuk regularisasi
    TARGET_MODULES  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

    # === Training — dikurangi epoch, naikkan regularisasi ===
    PER_DEVICE_BATCH_SIZE       = 1
    GRADIENT_ACCUMULATION_STEPS = 8          # effective batch = 8
    NUM_EPOCHS                  = 5          # dari 10 → val loss naik sejak epoch 3
    LEARNING_RATE               = 1e-4
    WEIGHT_DECAY                = 0.05       # dari 0.01 → regularisasi lebih kuat
    WARMUP_RATIO                = 0.05
    LR_SCHEDULER                = "cosine"

    # === Eval & Save ===
    EVAL_STRATEGY             = "epoch"
    SAVE_STRATEGY             = "epoch"
    LOGGING_STEPS             = 2
    SAVE_TOTAL_LIMIT          = 2            # dari 3 → hemat storage
    EARLY_STOPPING_PATIENCE   = 2            # dari 3 → stop lebih cepat
    EARLY_STOPPING_THRESHOLD  = 0.001

    # === Split ratio ===
    TEST_SIZE       = 0.1
    VALIDATION_SIZE = 0.1

    # === Paths ===
    DATASET_DIR     = DATASET_DIR
    SFT_SAVE_PATH   = os.path.join(OUTPUT_DIR, "prompt-expander-lora")
    CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "checkpoints")

os.makedirs(Config.SFT_SAVE_PATH, exist_ok=True)
os.makedirs(Config.CHECKPOINT_PATH, exist_ok=True)

print("⭐ Konfigurasi (anti-overfitting):")
print(f"   Model           : {Config.MODEL_NAME}")
print(f"   Max seq length  : {Config.MAX_SEQ_LENGTH}")
print(f"   LoRA r / alpha  : {Config.LORA_R} / {Config.LORA_ALPHA}")
print(f"   Effective batch : {Config.PER_DEVICE_BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS}")
print(f"   Epochs          : {Config.NUM_EPOCHS} (+ early stop patience={Config.EARLY_STOPPING_PATIENCE})")
print(f"   Learning rate   : {Config.LEARNING_RATE}")
print(f"   Weight decay    : {Config.WEIGHT_DECAY}")
print(f"   Output dir      : {Config.SFT_SAVE_PATH}")

## 3. Load Model + Tokenizer

In [ ]:
print(f"📦 Loading {Config.MODEL_NAME} ...")

tokenizer = AutoTokenizer.from_pretrained(
    Config.MODEL_NAME,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    Config.MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

print(f"✅ Model loaded (bf16 + gradient checkpointing)")
print(f"   device_map : auto")


## 4. Merge & Load Dataset

In [ ]:
import glob

# Auto-scan semua .json di DATASET_DIR dan semua subfoldernya
json_files = sorted(glob.glob(
    os.path.join(Config.DATASET_DIR, "**", "*.json"),
    recursive=True
))

# Exclude merged file (hasil generate notebook ini sendiri)
EXCLUDE = {"train_dataset_merged.json", "train_dataset.json", "train_dataset_synthetic.json"}
json_files = [
    f for f in json_files
    if os.path.basename(f) not in EXCLUDE
]

print(f"📂 Ditemukan {len(json_files)} file JSON:")
for f in json_files:
    rel = os.path.relpath(f, Config.DATASET_DIR)
    print(f"   - {rel}")

all_data = []
for fpath in json_files:
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        rel = os.path.relpath(fpath, Config.DATASET_DIR)
        if isinstance(data, list) and data and "messages" in data[0]:
            print(f"   ✅ {rel}: {len(data)} contoh")
            all_data.extend(data)
        else:
            print(f"   ⚠️  {rel}: skip (bukan format SFT)")
    except Exception as e:
        print(f"   ❌ {fpath}: {e}")

assert all_data, (
    f"❌ Tidak ada dataset valid di {Config.DATASET_DIR}"
)

# Deduplicate
seen = set()
deduped = []
for item in all_data:
    key = json.dumps(item, sort_keys=True, ensure_ascii=False)
    if key not in seen:
        seen.add(key)
        deduped.append(item)

print(f"\n📦 Total raw         : {len(all_data)} contoh")
print(f"📦 Setelah deduplikat: {len(deduped)} contoh")

merged_path = os.path.join(
    Config.DATASET_DIR, "train_dataset_merged.json"
)
with open(merged_path, "w", encoding="utf-8") as f:
    json.dump(deduped, f, ensure_ascii=False, indent=2)
print(f"💾 Merged → {merged_path}")

all_data = deduped

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_chat, remove_columns=["messages"])
print(f"✅ Dataset siap: {len(dataset)} contoh")
print(f"\n📝 Preview contoh pertama (200 char):\n{dataset[0]['text'][:200]}...")

## 5. Train / Validation / Test Split

In [ ]:
# Split: train (80%) / val (10%) / test (10%)
split1 = dataset.train_test_split(test_size=Config.TEST_SIZE, seed=Config.SEED)
test_dataset = split1["test"]

split2 = split1["train"].train_test_split(
    test_size = Config.VALIDATION_SIZE / (1 - Config.TEST_SIZE),
    seed      = Config.SEED,
)
train_dataset = split2["train"]
val_dataset   = split2["test"]

print(f"📊 Train : {len(train_dataset)}")
print(f"📊 Val   : {len(val_dataset)}")
print(f"📊 Test  : {len(test_dataset)}")

## 6. Apply LoRA (PEFT)

In [ ]:
lora_config = LoraConfig(
    task_type    = TaskType.CAUSAL_LM,
    r            = Config.LORA_R,
    lora_alpha   = Config.LORA_ALPHA,
    lora_dropout = Config.LORA_DROPOUT,
    target_modules = Config.TARGET_MODULES,
    bias         = "none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied")
print(f"   - LoRA rank     : {Config.LORA_R}")
print(f"   - LoRA alpha    : {Config.LORA_ALPHA}")
print(f"   - Target modules: {Config.TARGET_MODULES}")


## 7. Training Arguments + Trainer

In [ ]:
total_steps  = max(1, (len(train_dataset) // (Config.PER_DEVICE_BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS)) * Config.NUM_EPOCHS)
warmup_steps = int(total_steps * Config.WARMUP_RATIO)

print(f"📊 Total steps  : {total_steps}")
print(f"📊 Warmup steps : {warmup_steps}")

training_args = SFTConfig(
    output_dir                   = Config.CHECKPOINT_PATH,
    logging_steps                = Config.LOGGING_STEPS,
    per_device_train_batch_size  = Config.PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size   = Config.PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps  = Config.GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs             = Config.NUM_EPOCHS,
    learning_rate                = Config.LEARNING_RATE,
    weight_decay                 = Config.WEIGHT_DECAY,
    warmup_ratio                 = Config.WARMUP_RATIO,
    lr_scheduler_type            = Config.LR_SCHEDULER,
    optim                        = "adamw_8bit",
    eval_strategy                = Config.EVAL_STRATEGY,
    save_strategy                = Config.SAVE_STRATEGY,
    save_total_limit             = Config.SAVE_TOTAL_LIMIT,
    load_best_model_at_end       = True,
    metric_for_best_model        = "eval_loss",
    greater_is_better            = False,
    fp16                         = not torch.cuda.is_bf16_supported(),
    bf16                         = torch.cuda.is_bf16_supported(),
    dataset_text_field           = "text",
    packing                      = False,
    report_to                    = "none",
    seed                         = Config.SEED,
    data_seed                    = Config.SEED,
    remove_unused_columns        = False,
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    callbacks        = [
        EarlyStoppingCallback(
            early_stopping_patience  = Config.EARLY_STOPPING_PATIENCE,
            early_stopping_threshold = Config.EARLY_STOPPING_THRESHOLD,
        ),
    ],
)

print("✅ Trainer ready")


## 8. Train!

In [ ]:
print("🏋️  Mulai training...\n")
trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("🎉 Training selesai!")
print("=" * 60)
print(f"   Training loss final : {trainer_stats.training_loss:.4f}")
print(f"   Training time (s)   : {trainer_stats.metrics.get('train_runtime', 0):.1f}")

## 9. Evaluasi di Test Set

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
print("📊 Test metrics:")
for k, v in test_metrics.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

## 10. Save Model + Tokenizer

In [ ]:
print(f"💾 Saving LoRA adapter ke {Config.SFT_SAVE_PATH} ...")
model.save_pretrained(Config.SFT_SAVE_PATH)
tokenizer.save_pretrained(Config.SFT_SAVE_PATH)

# Save training config
config_dump = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
with open(f"{Config.SFT_SAVE_PATH}/training_config.json", "w") as f:
    json.dump(config_dump, f, indent=2, default=str)

print(f"✅ Saved: {Config.SFT_SAVE_PATH}")
print(f"   - adapter_model.safetensors")
print(f"   - tokenizer.json")
print(f"   - training_config.json")

## 11. Quick Inference Test

In [ ]:
model.eval()
test_input = "Buatkan game edukasi: mata_pelajaran=Fisika, jenjang=SMA-10, topik=gerak parabola"
messages = [{"role": "user", "content": test_input}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
print("Generating...")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))


## 12. Download Hasil ke Lokal (Colab only)

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    zip_path = "/content/prompt-expander-lora.zip"
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', Config.SFT_SAVE_PATH)
    print(f"📦 Zipped to {zip_path}")
    files.download(zip_path)
else:
    print(f"💡 Model tersimpan di: {Config.SFT_SAVE_PATH}")